# AgriDrone — Phase 2 training (Colab GPU)

This notebook is deliberately **thin**: all real logic lives in the
`agridrone.train` module so it stays version-controlled and unit-tested. The
notebook only wires Colab together — mount Drive, get the repo, install the
stack, and call the training stages.

**Before running:** set the runtime to a GPU (Runtime -> Change runtime type ->
T4 GPU), and make sure the processed PlantDoc dataset is cached to your Drive
(run `make data` locally, then `python -m agridrone.data --drive-cache \
/content/drive/MyDrive/agridrone/plantdoc` once with Drive mounted, or sync the
folder manually).

## 1. Mount Google Drive (cached dataset lives here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo
Clones the default branch (`main`). Point at a different fork/branch by editing `REPO_URL`.

In [ ]:
import os

REPO_URL = 'https://github.com/007-SARANG/agridrone.git'
REPO_DIR = '/content/agridrone'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR

## 3. Install the full ML stack (pinned versions)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)

## 4. Point the config at the Drive-cached dataset
Edit `configs/train.yaml` so `data.drive_dataset_dir` (and, if pretraining,
`pretrain.dataset_dir`) point at your Drive folders. You can also override in
memory below without editing the file.

In [ ]:
from agridrone.config import load_config

cfg = load_config('configs/train.yaml')
cfg['data']['drive_dataset_dir'] = '/content/drive/MyDrive/agridrone/plantdoc'
# Optional: enable PlantVillage backbone pretrain by pointing at an ImageFolder dir.
# cfg['pretrain']['dataset_dir'] = '/content/drive/MyDrive/agridrone/plantvillage'
from agridrone.train import resolve_dataset_yaml
print('dataset.yaml ->', resolve_dataset_yaml(cfg))

## 5. (Optional) Pretrain the backbone on PlantVillage
Skips itself if `pretrain.enabled` is false or `pretrain.dataset_dir` is empty.

In [ ]:
from agridrone.train import pretrain_backbone
backbone = pretrain_backbone(cfg)
print('backbone:', backbone)

## 6. Fine-tune the detector on PlantDoc
Uses the weighted-oversampling dataloader (rare classes seen more often) when
`weighting.enabled` is true.

In [ ]:
from agridrone.train import finetune_detector
results = finetune_detector(cfg, backbone=backbone)

## 7. (Optional) Train the YOLOv8 baseline for comparison

In [ ]:
# from agridrone.train import finetune_detector
# finetune_detector(cfg, baseline=True)  # vanilla, no weighted sampling

## 8. Evaluate on TEST -> per-class AP + mAP table

In [ ]:
from agridrone.train import evaluate
best = f"{cfg['output']['runs_dir']}/plantdoc_finetune/weights/best.pt"
metrics = evaluate(cfg, best)
print(open(cfg['output']['results_table']).read())